# Run the FL poisoning experiments on Colab (GPU)

This notebook:

1. mounts Google Drive and clones the repo on branch `fix-pipeline`,
2. installs the requirements,
3. runs a short GPU smoke test (1 seed, 2 rounds) and **stops if it fails**,
4. runs the full experiment **one seed at a time**, writing every run to Drive,
5. builds `summary.json` and the plot from the saved runs.

**Before running:** *Runtime → Change runtime type → GPU*, then *Runtime → Run all*.

**Resuming:** every finished (seed, attack, defense, z) run is saved as its own JSON file in
`RESULTS_DIR`. If the session disconnects, just run all cells again: finished runs are skipped.
Runs produced with different settings (e.g. another number of rounds) are refused rather than
mixed; point `RESULTS_DIR` to a new folder when you change settings.

**Reproducibility note:** runs are seeded, but GPU kernels are not bit-identical to CPU,
so numbers from Colab will differ slightly from local CPU runs with the same seed.

In [ ]:
# ---- Settings ----
REPO_URL = 'https://github.com/StrnMo/federated-learning-stealthy-attack.git'
BRANCH = 'fix-pipeline'
SEEDS = [0, 1, 2, 3, 4]
NUM_ROUNDS = 30

DRIVE_DIR = '/content/drive/MyDrive/fl-stealthy-attack'
RESULTS_DIR = f'{DRIVE_DIR}/results'   # one JSON per run + summary.json
PLOTS_DIR = f'{DRIVE_DIR}/plots'
DATA_DIR = f'{DRIVE_DIR}/medmnist'     # dataset cache, so BloodMNIST is downloaded only once

REPO_DIR = '/content/federated-learning-stealthy-attack'
SMOKE_DIR = '/content/smoke_test'      # local and wiped each time, so the smoke test always really runs

In [ ]:
import os
import subprocess
import sys

def run(cmd, cwd=None, env=None):
    """Run a command, stream its output, and raise if it fails (this stops "Run all")."""
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'Command failed (exit code {proc.returncode}): {" ".join(cmd)}')

## 1. Check the GPU and mount Google Drive

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU, then run again'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
for d in (RESULTS_DIR, PLOTS_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)

## 2. Clone the repo on `fix-pipeline` and install requirements

In [ ]:
if not os.path.isdir(REPO_DIR):
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    # Reuse an earlier clone in this VM, updated to the latest commit of the branch
    run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH])
    run(['git', '-C', REPO_DIR, 'checkout', '-B', BRANCH, f'origin/{BRANCH}'])
run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'])

run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO_DIR)

ENV = {**os.environ, 'MEDMNIST_ROOT': DATA_DIR}
SCRIPT = 'experiments/run_experiment.py'

## 3. GPU smoke test (1 seed, 2 rounds)

Runs every configuration for 2 rounds into a local scratch folder and checks that it ran on the
GPU and produced finite outputs. If anything fails, this cell raises and *Run all* stops here,
before the long run starts.

In [ ]:
import glob
import json
import shutil

shutil.rmtree(SMOKE_DIR, ignore_errors=True)
run([sys.executable, SCRIPT, '--seeds', '0', '--rounds', '2',
     '--results-dir', f'{SMOKE_DIR}/results', '--plots-dir', f'{SMOKE_DIR}/plots'],
    cwd=REPO_DIR, env=ENV)

sys.path.insert(0, f'{REPO_DIR}/experiments')
import run_experiment as rx
files = sorted(glob.glob(f'{SMOKE_DIR}/results/seed0_*.json'))
assert len(files) == len(rx.CONFIGS), f'expected {len(rx.CONFIGS)} run files, found {len(files)}'
for path in files:
    r = json.load(open(path))
    name = os.path.basename(path)
    assert r['config']['environment']['device'] == 'cuda', f'{name} did not run on the GPU'
    assert len(r['history']) == 2, f'{name} did not complete 2 rounds'
    assert r['test']['nonfinite_samples'] == 0, f'{name} produced non-finite logits'
assert os.path.exists(f'{SMOKE_DIR}/plots/experiment_results.png'), 'smoke-test plot missing'
print(f'Smoke test passed: {len(files)} runs on', r['config']['environment']['gpu'])

## 4. Full experiment, one seed at a time

Each seed runs every configuration and saves each run to `RESULTS_DIR` on Drive as soon as it
finishes. Re-running this cell skips runs that already exist.

In [ ]:
for seed in SEEDS:
    print(f'\n========== Seed {seed} ==========')
    run([sys.executable, SCRIPT, '--seeds', str(seed), '--rounds', str(NUM_ROUNDS),
         '--results-dir', RESULTS_DIR, '--skip-summary'],
        cwd=REPO_DIR, env=ENV)

## 5. Summary and plot from the saved results

Trains nothing: reads every run from `RESULTS_DIR` (failing if any is missing) and writes
`summary.json` there and `experiment_results.png` to `PLOTS_DIR`.

In [ ]:
run([sys.executable, SCRIPT, '--seeds', *map(str, SEEDS), '--rounds', str(NUM_ROUNDS),
     '--results-dir', RESULTS_DIR, '--plots-dir', PLOTS_DIR, '--summary-only'],
    cwd=REPO_DIR, env=ENV)

from IPython.display import Image, display
display(Image(f'{PLOTS_DIR}/experiment_results.png'))

In [ ]:
import pandas as pd

summary = json.load(open(f'{RESULTS_DIR}/summary.json'))['summary']
def fmt(m):
    return '' if m is None else f"{m['mean']:.4f} ± {m['std']:.4f}"
table = pd.DataFrame({
    name: {
        'test accuracy': fmt(s['test_accuracy']),
        'test balanced accuracy': fmt(s['test_balanced_accuracy']),
        'attack magnitude (L2)': fmt(s['stealthiness']),
        'detection TPR': fmt(s['detection_tpr']),
        'detection FPR': fmt(s['detection_fpr']),
        'seeds': s['num_seeds'],
    }
    for name, s in summary.items()
}).T
table